# Proyecto Data Mining - CRISP-DM

**Integrantes:** Flavio Henriquez / Bastian Bianchi  
**Docente:** Italo Bonnet

## 1. Entendimiento del Negocio

En esta fase se define el contexto del problema, la utilidad del análisis, las variables más relevantes y el enfoque general del proyecto dentro de la metodología CRISP-DM.

### 1.1 Contexto

Chile presenta una gran diversidad climática y geográfica: desde zonas altiplánicas y desérticas en el norte hasta sectores mediterráneos y cordilleranos en la zona central. En este escenario, analizar las precipitaciones es relevante para la gestión hídrica, la planificación agrícola, la prevención de eventos extremos y la toma de decisiones territoriales.

El dataset utilizado contiene registros mensuales de agua caída en estaciones meteorológicas chilenas, por lo que permite estudiar cómo cambia la precipitación según la ubicación, la altura y el momento del año. A partir de este análisis se busca comprender el comportamiento de las lluvias y dejar una base útil para futuras tareas de predicción.

El desarrollo del proyecto se estructura con la metodología CRISP-DM, por lo que esta primera fase corresponde al entendimiento del negocio y a la definición del problema desde una perspectiva aplicada a las precipitaciones en Chile.

En coherencia con los contenidos de la unidad, el foco inicial no está en construir de inmediato un modelo final, sino en definir con claridad el objetivo del análisis, el valor que puede aportar y los criterios que orientarán las siguientes etapas de entendimiento, limpieza y preprocesamiento.


### 1.2 Datos relevantes

El conjunto de datos considera 552 registros, correspondientes a 10 estaciones meteorológicas ubicadas en 4 regiones de Chile, con observaciones mensuales entre 2021 y 2025.

Las variables disponibles pueden resumirse de la siguiente manera:

| Variable | Descripción | Tipo |
| --- | --- | --- |
| `codigo_estacion` | Identificador único de la estación meteorológica. | Categórica nominal |
| `nombre_estacion` | Nombre de la estación meteorológica. | Categórica nominal |
| `region` | Código numérico de la región asociada a la estación. | Categórica nominal |
| `nombre_region` | Nombre de la región donde se ubica la estación. | Categórica nominal |
| `altura` | Altitud de la estación sobre el nivel del mar. | Numérica discreta |
| `latitud` | Coordenada geográfica de latitud de la estación. | Numérica continua |
| `longitud` | Coordenada geográfica de longitud de la estación. | Numérica continua |
| `anio` | Año en que fue registrado el dato. | Numérica discreta |
| `mes` | Mes en que fue registrado el dato. | Numérica discreta |
| `agua_caida` | Cantidad de precipitación registrada en el mes, medida en milímetros. | Numérica continua |

La variable principal de interés es `agua_caida`, ya que representa el nivel de precipitación que se desea analizar y eventualmente estimar.


### 1.3 Hipótesis y tesis consideradas

Se consideran las siguientes hipótesis de trabajo:
- El mes del año influye directamente en la cantidad de precipitación debido a la estacionalidad climática.
- La ubicación geográfica de la estación (`latitud`, `longitud` y `region`) explica parte importante de la variación de `agua_caida`.
- La altura de la estación puede estar relacionada con diferencias en los niveles de precipitación.
- Existen estaciones con comportamientos históricos distintos, por lo que `codigo_estacion` o `nombre_estacion` pueden aportar valor explicativo.
- Las regiones del norte presentarán menores niveles de precipitación que zonas más centrales del conjunto analizado.
- Puede haber variaciones entre años por efectos climáticos interanuales.


### 1.4 KPI's relevantes

Considerando que esta primera unidad está orientada al entendimiento y preparación de los datos, los criterios de éxito más relevantes para esta fase serán:
- Identificar correctamente las variables disponibles y su utilidad dentro del problema.
- Reconocer la cobertura temporal y geográfica del dataset para entender su alcance real.
- Detectar nulos, posibles outliers y patrones iniciales que puedan afectar el análisis posterior.
- Dejar bien formulada la variable objetivo y el enfoque analítico del proyecto.

Si más adelante el proyecto avanza hacia una etapa predictiva, los KPI técnicos más pertinentes serán:
- MAE (Error Absoluto Medio): mide el error promedio en milímetros y es fácil de interpretar.
- RMSE (Raíz del Error Cuadrático Medio): penaliza con mayor fuerza los errores grandes en meses de lluvia intensa.
- R2: indica qué proporción de la variabilidad de la precipitación logra explicar el modelo.
- Análisis de errores por región y por estación: permite evaluar si el desempeño es estable en distintos contextos geográficos.

A nivel de negocio, un criterio central será lograr una caracterización clara y útil del comportamiento de las precipitaciones; y, en una etapa predictiva posterior, el KPI más interpretable sería el `MAE`.


### 1.5 Problema de negocio (Análisis y proyección predictiva)

El problema de negocio consiste en comprender, analizar y estimar la cantidad de agua caída mensual en estaciones meteorológicas de Chile a partir de variables geográficas y temporales. Esto puede apoyar decisiones relacionadas con gestión de recursos hídricos, planificación territorial, monitoreo climático y preparación frente a periodos de sequía o precipitaciones intensas.

Desde el punto de vista de data mining, en esta etapa el enfoque principal es descriptivo y exploratorio, buscando descubrir patrones relevantes en los datos. Como proyección del proyecto, el problema puede formularse posteriormente como una tarea de regresión supervisada, donde la variable a explicar es `agua_caida` y los predictores iniciales corresponden a la estación, su ubicación, la altura y la fecha del registro.


## 2. Entendimiento de los Datos

En esta fase se realiza una exploración inicial del dataset para identificar tipos de variables, presencia de nulos, outliers, estadísticas descriptivas, correlaciones y patrones relevantes en las precipitaciones.

### 2.1 Identificar Tipos de variables/atributos

En esta subsecci?n distinguimos entre el tipo t?cnico detectado por `pandas` y el tipo anal?tico que realmente representa cada atributo. Esto es importante porque varias columnas se almacenan como n?meros, pero conceptualmente funcionan como identificadores o categor?as.


In [ ]:
# Si necesitas instalar dependencias en este notebook:
# %pip install pandas numpy matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 120

def heatmap_df(data, title, cmap="Blues", figsize=(8, 5), decimals=2):
    fig, ax = plt.subplots(figsize=figsize)
    matrix = data.astype(float).values
    im = ax.imshow(matrix, cmap=cmap, aspect="auto")
    ax.set_xticks(range(len(data.columns)))
    ax.set_xticklabels(data.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(data.index)))
    ax.set_yticklabels(data.index)
    for row in range(data.shape[0]):
        for col in range(data.shape[1]):
            value = matrix[row, col]
            ax.text(col, row, f"{value:.{decimals}f}", ha="center", va="center", fontsize=8)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()

df = pd.read_csv("dataset_precipitaciones_chile.csv")
print(f"Dimension del dataset: {df.shape[0]} filas x {df.shape[1]} columnas")
display(df.head())


In [ ]:
resumen_base = pd.DataFrame({
    "dtype_pandas": df.dtypes.astype(str),
    "valores_unicos": df.nunique(),
    "nulos": df.isnull().sum()
})

display(resumen_base)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
resumen_base["valores_unicos"].sort_values().plot(kind="barh", ax=ax, color="#2563eb")
ax.set_title("Cantidad de valores unicos por variable")
ax.set_xlabel("Valores unicos")
ax.set_ylabel("Variable")
plt.tight_layout()
plt.show()


In [ ]:
clasificacion_variables = pd.DataFrame({
    "variable": ["codigo_estacion", "nombre_estacion", "region", "nombre_region", "altura", "latitud", "longitud", "anio", "mes", "agua_caida"],
    "tipo_analitico": [
        "Categ?rica nominal", "Categ?rica nominal", "Categ?rica nominal", "Categ?rica nominal",
        "Num?rica discreta", "Num?rica continua", "Num?rica continua", "Num?rica discreta",
        "Num?rica discreta", "Num?rica continua"
    ],
    "rol_en_el_proyecto": [
        "Identificador de estaci?n", "Identificador descriptivo", "C?digo territorial", "Categor?a territorial",
        "Predictor geogr?fico", "Predictor geogr?fico", "Predictor geogr?fico", "Predictor temporal",
        "Predictor temporal", "Variable objetivo"
    ]
})

display(clasificacion_variables)


**An?lisis.** El dataset contiene 10 variables: dos categ?ricas de texto, dos c?digos num?ricos con sentido categ?rico y seis variables num?ricas asociadas a ubicaci?n, tiempo y precipitaci?n. La variable objetivo es `agua_caida`. Un punto importante es que `codigo_estacion` y `region` no deben interpretarse como magnitudes continuas, porque sus valores num?ricos representan etiquetas y no distancia real entre categor?as.


### 2.2 Identificar Nulo

Antes de aplicar cualquier transformaci?n conviene revisar si existen valores faltantes. En esta etapa no buscamos todav?a imputar, sino diagnosticar si el problema est? presente y medir su magnitud.


In [ ]:
tabla_nulos = pd.DataFrame({
    "nulos": df.isnull().sum(),
    "porcentaje_nulos": (df.isnull().mean() * 100).round(2)
}).sort_values(["nulos", "porcentaje_nulos"], ascending=False)

display(tabla_nulos)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
tabla_nulos["nulos"].plot(kind="bar", ax=ax, color="#94a3b8")
ax.set_title("Valores nulos por variable")
ax.set_xlabel("Variable")
ax.set_ylabel("Cantidad de nulos")
ax.set_ylim(0, max(1, int(tabla_nulos["nulos"].max()) + 1))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


**An?lisis.** El dataset no presenta valores nulos en ninguna columna. Esto simplifica la preparaci?n de datos, ya que en esta fase no se requiere imputaci?n ni eliminaci?n de registros por informaci?n faltante. En consecuencia, el an?lisis puede concentrarse en la distribuci?n de `agua_caida` y en las diferencias entre estaciones y regiones.


### 2.3 Identificar Outliers

La detecci?n de outliers debe hacerse con criterio de negocio y no de forma autom?tica. En este caso primero observamos la distribuci?n de la variable objetivo, porque una gran cantidad de ceros puede alterar la interpretaci?n de m?todos tradicionales como IQR.


In [ ]:
distribucion_lluvia = pd.DataFrame({
    "indicador": ["Meses sin precipitaci?n", "Meses con precipitaci?n"],
    "porcentaje": [
        round((df["agua_caida"] == 0).mean() * 100, 2),
        round((df["agua_caida"] > 0).mean() * 100, 2)
    ]
})

display(distribucion_lluvia)


In [ ]:
q1 = df["agua_caida"].quantile(0.25)
q3 = df["agua_caida"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

resumen_iqr = pd.DataFrame({
    "Q1": [q1],
    "Q3": [q3],
    "IQR": [iqr],
    "L?mite inferior": [lower],
    "L?mite superior": [upper],
    "Registros sobre el l?mite superior": [(df["agua_caida"] > upper).sum()]
}).round(2)

display(resumen_iqr)


In [ ]:
extremos_lluvia = (
    df.loc[df["agua_caida"] > upper, ["nombre_estacion", "nombre_region", "anio", "mes", "agua_caida"]]
      .sort_values("agua_caida", ascending=False)
      .head(10)
)

display(extremos_lluvia)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["agua_caida"], bins=30, color="#0f766e", edgecolor="white")
axes[0].set_title("Distribucion de agua_caida")
axes[0].set_xlabel("Milimetros")
axes[0].set_ylabel("Frecuencia")

axes[1].boxplot(df["agua_caida"], vert=False)
axes[1].set_title("Boxplot de agua_caida")
axes[1].set_xlabel("Milimetros")

plt.tight_layout()
plt.show()


In [ ]:
regiones_box = sorted(df["nombre_region"].unique())
datos_box = [df.loc[df["nombre_region"] == region, "agua_caida"] for region in regiones_box]

fig, ax = plt.subplots(figsize=(10, 4))
ax.boxplot(datos_box, labels=regiones_box, vert=False)
ax.set_title("Boxplot de agua_caida por region")
ax.set_xlabel("Milimetros")
ax.set_ylabel("Region")
plt.tight_layout()
plt.show()


**An?lisis.** El 75.72% de los registros presenta `agua_caida = 0`, por lo que la distribuci?n est? fuertemente concentrada en cero. Bajo este escenario, el m?todo IQR produce `Q1 = 0`, `Q3 = 0` e `IQR = 0`, haciendo que cualquier valor positivo aparezca como outlier. Por eso, en este dataset no conviene interpretar autom?ticamente los positivos como errores. El boxplot general confirma una distribuci?n muy asim?trica y el boxplot por regi?n muestra que la mayor dispersi?n se concentra principalmente en Metropolitana de Santiago y Arica y Parinacota. Los valores m?s altos se concentran en Quinta Normal, Visviri y Putre, y son coherentes con eventos reales de precipitaci?n intensa en sus respectivas zonas.


### 2.4 Transformaciones

Antes de codificar o ajustar variables, conviene definir un dataset de trabajo m?s limpio. En esta etapa eliminamos duplicidades de representaci?n para quedarnos con columnas m?s interpretables desde el punto de vista anal?tico.


In [ ]:
df_trabajo = df[[
    "nombre_estacion",
    "nombre_region",
    "altura",
    "latitud",
    "longitud",
    "anio",
    "mes",
    "agua_caida"
]].copy()

print("Columnas seleccionadas para el trabajo anal?tico:")
display(pd.DataFrame({"columnas": df_trabajo.columns}))
display(df_trabajo.head())


**An?lisis.** En esta selecci?n se conservan las variables con mayor valor interpretativo. `codigo_estacion` se excluye porque replica la identidad de `nombre_estacion`, y `region` se omite porque entrega la misma informaci?n que `nombre_region`, pero de forma menos descriptiva. De esta manera, el conjunto de trabajo queda m?s limpio y m?s f?cil de explicar.


#### 2.4.1 A numéricos (Encoder)

Como `nombre_estacion` y `nombre_region` son variables nominales, si en etapas posteriores se quieren usar en modelos num?ricos deben transformarse sin imponer un orden artificial. Para ello utilizamos *one-hot encoding*.


In [ ]:
df_encoder = pd.get_dummies(
    df_trabajo,
    columns=["nombre_estacion", "nombre_region"],
    drop_first=False,
    dtype=int
)

print(f"Shape original: {df_trabajo.shape}")
print(f"Shape despu?s del encoding: {df_encoder.shape}")
display(df_encoder.iloc[:5, :12])


**An?lisis.** Se utiliza *one-hot encoding* porque las estaciones y regiones no tienen un orden natural. Si se aplicara `Label Encoding`, el modelo podr?a interpretar err?neamente que una categor?a es ?mayor? o ?menor? que otra. La codificaci?n binaria evita ese problema y conserva la naturaleza nominal de las variables.


#### 2.4.2 Tratamiento de Nulos

Aunque ya comprobamos que no existen nulos, conviene validarlo nuevamente sobre el dataset de trabajo, porque cualquier selecci?n o transformaci?n podr?a modificar esa condici?n.


In [ ]:
nulos_post_seleccion = df_trabajo.isnull().sum().sum()
print(f"Total de valores nulos en el dataset de trabajo: {nulos_post_seleccion}")


**An?lisis.** El conjunto de trabajo sigue sin valores faltantes, por lo que no se requiere imputaci?n. Esto es positivo porque evita introducir sesgos artificiales en una base que ya viene completa.


#### 2.4.3 Tratamiento de Outliers

Dado que los m?ximos de precipitaci?n pueden corresponder a eventos clim?ticos reales, en esta fase no se eliminan registros extremos. En lugar de borrar informaci?n, se crea una transformaci?n logar?tmica de apoyo para reducir la asimetr?a de la variable objetivo.


In [ ]:
df_trabajo["agua_caida_log"] = np.log1p(df_trabajo["agua_caida"])

comparacion_escala = df_trabajo[["agua_caida", "agua_caida_log"]].describe().T.round(2)
display(comparacion_escala)


**An?lisis.** La estrategia elegida no elimina extremos, sino que conserva la variabilidad original de la lluvia y agrega una versi?n m?s estable (`agua_caida_log`) para an?lisis complementarios. Esto es m?s coherente con un problema clim?tico, donde los m?ximos pueden ser precisamente los casos m?s interesantes.


### 2.5 Análisis estadísticos básicos 

Una vez revisada la calidad de los datos, se realiza una descripci?n estad?stica b?sica. Primero se resumen las variables num?ricas y luego se observa c?mo cambia la precipitaci?n seg?n la regi?n.


In [ ]:
estadisticas_numericas = df[["altura", "latitud", "longitud", "anio", "mes", "agua_caida"]].describe().T.round(2)
display(estadisticas_numericas)


In [ ]:
resumen_region = (
    df.groupby("nombre_region")["agua_caida"]
      .agg(["count", "mean", "median", "sum", "max"])
      .round(2)
      .sort_values("sum", ascending=False)
)

display(resumen_region)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

resumen_region["mean"].sort_values(ascending=False).plot(kind="bar", ax=axes[0], color="#2563eb")
axes[0].set_title("Precipitacion promedio por region")
axes[0].set_xlabel("Region")
axes[0].set_ylabel("Promedio de mm")
axes[0].tick_params(axis="x", rotation=45)

resumen_region["sum"].sort_values(ascending=False).plot(kind="bar", ax=axes[1], color="#0f766e")
axes[1].set_title("Precipitacion acumulada por region")
axes[1].set_xlabel("Region")
axes[1].set_ylabel("Suma de mm")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


**An?lisis.** La estad?stica descriptiva confirma una distribuci?n muy sesgada de `agua_caida`: la mediana es 0 mm, mientras que el promedio es 4.87 mm y el m?ximo alcanza 183.1 mm. A nivel regional, Metropolitana de Santiago presenta la media m?s alta (20.20 mm) y el mayor acumulado total, seguida por Arica y Parinacota. Tarapac? y Antofagasta muestran valores considerablemente m?s bajos, lo que refuerza la heterogeneidad clim?tica del dataset.


### 2.6 Matriz de Correlación 

La correlaci?n inicial se calcula solo con variables num?ricas disponibles en bruto. Esta revisi?n sirve como una primera aproximaci?n lineal, aunque todav?a no incorpora de buena forma la naturaleza c?clica del mes ni la informaci?n categ?rica de regi?n o estaci?n.


In [ ]:
corr_inicial = df[["altura", "latitud", "longitud", "anio", "mes", "agua_caida"]].corr(numeric_only=True).round(3)
display(corr_inicial)
display(corr_inicial["agua_caida"].sort_values(ascending=False))


In [ ]:
heatmap_df(corr_inicial, "Matriz de correlacion inicial", cmap="coolwarm", figsize=(8, 5), decimals=2)


**An?lisis.** En la matriz inicial, las relaciones lineales con `agua_caida` son bajas o moderadas. La m?s visible es `latitud` (-0.241), seguida por `altura` (0.182) y `anio` (0.115). `mes` aparece con una relaci?n d?bil (-0.120), lo que sugiere que tratarlo como n?mero lineal no captura adecuadamente la estacionalidad real del fen?meno.


### 2.7 Patrones/comportamientos detectados

M?s all? de la correlaci?n lineal, en miner?a de datos tambi?n interesa identificar comportamientos concretos en el tiempo y en el espacio. Para eso revisamos la lluvia por mes, por a?o y por estaci?n meteorol?gica.


In [ ]:
lluvia_mensual = (
    df.groupby("mes")["agua_caida"]
      .agg(["mean", "median", "sum", "max"])
      .round(2)
)

display(lluvia_mensual)


In [ ]:
lluvia_anual = (
    df.groupby("anio")["agua_caida"]
      .agg(["mean", "median", "sum", "max"])
      .round(2)
)

display(lluvia_anual)


In [ ]:
lluvia_estacion = (
    df.groupby("nombre_estacion")["agua_caida"]
      .sum()
      .sort_values(ascending=False)
      .round(2)
      .to_frame("agua_acumulada")
)

display(lluvia_estacion)


In [ ]:
patron_estacion_mes = df.pivot_table(
    index="mes",
    columns="nombre_estacion",
    values="agua_caida",
    aggfunc="mean"
).round(2)

display(patron_estacion_mes)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

lluvia_mensual["mean"].plot(marker="o", ax=axes[0], color="#2563eb")
axes[0].set_title("Promedio mensual de precipitacion")
axes[0].set_xlabel("Mes")
axes[0].set_ylabel("Promedio de mm")

lluvia_anual["sum"].plot(kind="bar", ax=axes[1], color="#0f766e")
axes[1].set_title("Precipitacion acumulada por anio")
axes[1].set_xlabel("Anio")
axes[1].set_ylabel("Suma de mm")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()


In [ ]:
heatmap_df(patron_estacion_mes, "Promedio mensual por estacion", cmap="YlGnBu", figsize=(11, 6), decimals=1)


**An?lisis.** Aqu? aparecen dos reg?menes de lluvia distintos. En Quinta Normal, los montos m?s altos se concentran entre mayo y agosto, coherente con el patr?n mediterr?neo de la zona central. En cambio, Visviri, Putre y Colchane muestran sus m?ximos entre enero y marzo, lo que coincide con las precipitaciones estivales del altiplano. Tambi?n se observa que 2024 es el a?o con mayor acumulado total dentro del per?odo analizado.


### 2.8 Nuevas variables a partir de análisis

A partir de los patrones detectados, conviene crear variables que representen mejor la estacionalidad y la diferencia territorial del dataset. El objetivo no es agregar columnas por agregar, sino construir atributos con sentido anal?tico.


In [ ]:
estaciones = {
    12: "Verano", 1: "Verano", 2: "Verano",
    3: "Oto?o", 4: "Oto?o", 5: "Oto?o",
    6: "Invierno", 7: "Invierno", 8: "Invierno",
    9: "Primavera", 10: "Primavera", 11: "Primavera"
}

df_features = df_trabajo.copy()
df_features["estacion_anual"] = df_features["mes"].map(estaciones)
df_features["mes_sin"] = np.sin(2 * np.pi * df_features["mes"] / 12)
df_features["mes_cos"] = np.cos(2 * np.pi * df_features["mes"] / 12)
df_features["macrozona"] = np.where(
    df_features["nombre_region"].eq("Metropolitana de Santiago"),
    "Centro",
    "Norte"
)

display(df_features[["mes", "estacion_anual", "mes_sin", "mes_cos", "nombre_region", "macrozona"]].head(12))


**An?lisis.** Las nuevas variables intentan capturar informaci?n que el `mes` en bruto no representa bien. `estacion_anual` resume el calendario clim?tico, `mes_sin` y `mes_cos` transforman el mes en una estructura c?clica, y `macrozona` separa el comportamiento de la zona central respecto del norte. Estas variables son m?s coherentes con la l?gica clim?tica del problema.


### 2.9 Matriz de Correlación 

Con las nuevas variables disponibles, volvemos a revisar la correlaci?n para ver si aparecen relaciones m?s interpretables con `agua_caida`. En este caso se incluyen representaciones c?clicas y una variable territorial simplificada.


In [ ]:
corr_extendida = df_features[["altura", "latitud", "longitud", "anio", "mes_sin", "mes_cos", "agua_caida"]].copy()
corr_extendida["macrozona_centro"] = (df_features["macrozona"] == "Centro").astype(int)
corr_extendida["verano"] = (df_features["estacion_anual"] == "Verano").astype(int)
corr_extendida["invierno"] = (df_features["estacion_anual"] == "Invierno").astype(int)
corr_extendida["primavera"] = (df_features["estacion_anual"] == "Primavera").astype(int)

matriz_corr_extendida = corr_extendida.corr(numeric_only=True).round(3)
display(matriz_corr_extendida)
display(matriz_corr_extendida["agua_caida"].sort_values(ascending=False))


In [ ]:
heatmap_df(matriz_corr_extendida, "Matriz de correlacion con nuevas variables", cmap="coolwarm", figsize=(10, 7), decimals=2)


**An?lisis.** Con las variables derivadas, `macrozona_centro` pasa a ser la relaci?n positiva m?s alta con `agua_caida` (0.296), superando a las variables temporales originales. `latitud` sigue mostrando una relaci?n negativa relevante (-0.241), mientras que `verano` y `mes_sin` capturan una parte de la estacionalidad que antes se perd?a al usar solo `mes`. Aun as?, las correlaciones contin?an siendo moderadas, lo que indica que el fen?meno depende de interacciones m?s complejas que una sola relaci?n lineal.


### 2.10 Patrones/comportamientos detectados con nuevas variables

Finalmente, evaluamos si las variables nuevas ayudan a revelar patrones m?s claros. En particular, interesa verificar si la combinaci?n entre macrozona y estaci?n del a?o distingue mejor los reg?menes de precipitaci?n.


In [ ]:
patron_macrozona = (
    df_features.groupby(["macrozona", "estacion_anual"])["agua_caida"]
      .mean()
      .round(2)
      .unstack()
)

display(patron_macrozona)


In [ ]:
df_frecuencia = df.copy()
df_frecuencia["tiene_precipitacion"] = (df_frecuencia["agua_caida"] > 0).astype(int)

frecuencia_region = (
    df_frecuencia.groupby("nombre_region")["tiene_precipitacion"]
      .mean()
      .mul(100)
      .round(2)
      .to_frame("porcentaje_meses_con_lluvia")
)

display(frecuencia_region)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

patron_macrozona.T.plot(kind="bar", ax=axes[0], color=["#2563eb", "#f97316"])
axes[0].set_title("Precipitacion promedio por macrozona y estacion")
axes[0].set_xlabel("Estacion del ano")
axes[0].set_ylabel("Promedio de mm")
axes[0].legend(title="Macrozona")
axes[0].tick_params(axis="x", rotation=0)

frecuencia_region["porcentaje_meses_con_lluvia"].sort_values(ascending=False).plot(kind="bar", ax=axes[1], color="#16a34a")
axes[1].set_title("Frecuencia de meses con precipitacion por region")
axes[1].set_xlabel("Region")
axes[1].set_ylabel("Porcentaje")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


**An?lisis.** Las nuevas variables permiten separar con mayor claridad dos comportamientos clim?ticos. En la macrozona Centro, el promedio de precipitaci?n es mucho mayor en invierno (50.16 mm) que en verano (2.93 mm), mientras que en la macrozona Norte ocurre lo contrario: verano (8.13 mm) supera ampliamente a invierno (0.15 mm). Adem?s, Metropolitana de Santiago presenta lluvia en el 66.67% de sus meses registrados, frente a frecuencias bastante menores en Tarapac? (10.90%) y Antofagasta (15.00%). Esto confirma que la combinaci?n entre territorio y estacionalidad explica mejor el fen?meno que el mes tratado de forma lineal.


## 3. Transformación 

En esta fase se aplican las transformaciones necesarias para dejar los datos preparados para análisis posteriores, incluyendo codificación de variables, tratamiento de valores faltantes, manejo de outliers y ajustes derivados del preprocesamiento.

### 3.1 Escalamiento